# Online Retail Customer & Sales Analytics

## Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

## Loading Dataset

In [2]:
rfm = pd.read_csv("D:/Datasets/python_rfm_data.csv")

print(rfm.head())
print(rfm.columns)

rfm.columns = ['customer_id', 'recency', 'frequency', 'monetary']

   customer_id  recency  frequency  monetary
0        12346     5550          1  77183.60
1        12347     5227          7   4310.00
2        12348     5300          4   1797.24
3        12349     5243          1   1757.55
4        12350     5535          1    334.40
Index(['customer_id', 'recency', 'frequency', 'monetary'], dtype='object')


## Data Understanding

In [10]:
print("Shape:", rfm.shape)
rfm.info()
rfm.describe()

Shape: (4338, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4338 entries, 0 to 4337
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  4338 non-null   int64  
 1   recency      4338 non-null   int64  
 2   frequency    4338 non-null   int64  
 3   monetary     4338 non-null   float64
dtypes: float64(1), int64(3)
memory usage: 135.7 KB


,customer_id,recency,frequency,monetary
count,4338.000000,4338.000000,4338.000000,4338.000000
mean,15300.408022,5317.059474,4.272015,2054.266459
std,1721.808492,100.012264,7.697998,8989.230441
min,12346.000000,5225.000000,1.000000,3.750000
25%,13813.250000,5242.000000,1.000000,307.415000
50%,15299.500000,5275.000000,2.000000,674.485000
75%,16778.750000,5366.750000,5.000000,1661.740000
max,18287.000000,5598.000000,209.000000,280206.020000


## Checking Missing Values

In [11]:
rfm.isnull().sum()

customer_id    0
recency        0
frequency      0
monetary       0
dtype: int64

## Feature Engineering

In [12]:
rfm['AOV'] = rfm['monetary'] / rfm['frequency'].replace(0, 1)

## Creating Target (Churn)

In [19]:
threshold = rfm['recency'].quantile(0.7)

rfm['Churn'] = rfm['recency'].apply(lambda x: 1 if x > threshold else 0)

## Prepare Data

In [20]:
X = rfm[['frequency', 'monetary', 'AOV']]
y = rfm['Churn']

## TRAIN TEST SPLIT

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Logistic Regression

In [22]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
lr_prob = lr.predict_proba(X_test)[:, 1]

print("\n=== Logistic Regression ===")
print(classification_report(y_test, lr_pred))
print("ROC-AUC:", roc_auc_score(y_test, lr_prob))


=== Logistic Regression ===
              precision    recall  f1-score   support

           0       0.79      0.81      0.80       608
           1       0.52      0.50      0.51       260

    accuracy                           0.71       868
   macro avg       0.66      0.65      0.65       868
weighted avg       0.71      0.71      0.71       868

ROC-AUC: 0.7607097672064776


## Random Forest

In [23]:
rf = RandomForestClassifier(random_state=42)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("\n=== Random Forest ===")
print(classification_report(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_prob))


=== Random Forest ===
              precision    recall  f1-score   support

           0       0.75      0.76      0.76       608
           1       0.42      0.39      0.40       260

    accuracy                           0.65       868
   macro avg       0.58      0.58      0.58       868
weighted avg       0.65      0.65      0.65       868

ROC-AUC: 0.6866966093117409


## Feature Importance

In [24]:
importance = pd.Series(rf.feature_importances_, index=X.columns)

print("\nFeature Importance:")
print(importance.sort_values(ascending=False))


Feature Importance:
monetary     0.474206
AOV          0.411058
frequency    0.114735
dtype: float64


## Churn Probability

In [25]:
test_results = X_test.copy()
test_results['Churn_Probability'] = rf.predict_proba(X_test)[:, 1]
test_results['Actual'] = y_test.values

high_risk = test_results[test_results['Churn_Probability'] > 0.7]

print(high_risk.head())

      frequency  monetary     AOV  Churn_Probability  Actual
2841          1    159.00  159.00           0.960000       0
2105          1    305.75  305.75           0.948333       1
3645          1    160.57  160.57           0.950000       0
1270          1    265.85  265.85           0.890000       1
3528          2    206.34  103.17           0.890000       0


## Data Segmentation

In [26]:
rfm['Churn_Probability'] = rf.predict_proba(X)[:, 1]

rfm['Risk_Level'] = pd.cut(
    rfm['Churn_Probability'],
    bins=[0, 0.4, 0.7, 1],
    labels=['Low Risk', 'Medium Risk', 'High Risk'])

print(rfm[['customer_id', 'Churn_Probability', 'Risk_Level']].head())

   customer_id  Churn_Probability Risk_Level
0        12346               0.34   Low Risk
1        12347               0.00        NaN
2        12348               0.00        NaN
3        12349               0.04   Low Risk
4        12350               0.82  High Risk


## Saving output

In [28]:
rfm.to_csv("D:/Datasets/customer_churn_predictions.csv", index=False)

## Key Insights

 • Total revenue is highly concentrated, with a small percentage of customers
   contributing a large share

 • Majority of customers are low-frequency buyers, indicating weak retention

 • High recency customers (inactive for long periods) show significantly higher churn probability

 • Frequency and monetary value are strong indicators of customer loyalty

 • Average Order Value (AOV) combined with frequency provides deeper behavioral insight

 • Model enables probability-based segmentation of customers into Low, Medium, and High risk

 • High-risk customers represent a critical segment for targeted retention strategies

## Conclusion

 • RFM-based features effectively capture customer purchasing behavior

 • Machine learning model successfully predicts churn probability with realistic performance
 
 • Data-driven churn segmentation allows businesses to proactively identify at-risk customers

 • Revenue dependency on a small customer segment highlights retention risk

 • Improving retention of high-value and at-risk customers can significantly increase overall revenue

 • Using data-driven thresholds (instead of fixed assumptions) improved model reliability

• The project demonstrates how combining SQL, Python, and ML enables actionable business insights